In [25]:
import pandas as pd
import matplotlib.pyplot as plt

In [26]:
dataset_path = "dataset/dataset-uf100-430.csv"
df = pd.read_csv(dataset_path)

# replace NaN values in 'option' by 'default'
df['option'] = df['option'].fillna('default')
df = df.fillna(0)
# remove the ' ms' from the 'Time' column and convert it to float
df['Propagation'] = df['Propagation'] # - df["Skipped Propagation"] - df["Replayed Propagation"]


In [27]:
# add the scaled down columns (divided by 1000, 1000000) if they exist
for col in df.columns:
    if col in ['file', 'option']:
        continue
    # if numeric column
    if pd.api.types.is_numeric_dtype(df[col]):
        if df[col].max() > 1000000:
            df[col + ' x10^6'] = df[col] / 1000000
        if df[col].max() > 1000:
            df[col + ' x10^3'] = df[col] / 1000

In [28]:
# print(f"Average number of cross implication for decision filtered {filtered[filtered['option'] == '-gb']['Cross implication for decision'].mean()}")
# print(f"Average number of cross implication for decision df {df[df['option'] == '-gb']['Cross implication for decision'].mean()}")

In [29]:
options = df['option'].unique()

df_options = {opt: df[df['option'] == opt] for opt in options}

In [30]:
is_unsat = 'uuf' in dataset_path
n_vars = dataset_path.split('uf')[-1].split('-')[0]
if is_unsat:
    print("UNSAT", end=' ')
else:
    print("SAT", end=' ')
print(f"dataset with {int(len(df) / len(df_options))} problems with {n_vars} variables.")
columns = ["option"]
for col in df.columns:
    if col not in ['file', 'option']:
        columns.append(col + " mean")
        columns.append(col + " median")

stats = pd.DataFrame(columns=columns)
for opt, df in df_options.items():
    row = {}
    row['option'] = opt
    for col in df.columns:
        if col not in ['file', 'option']:
            row[col + " mean"] = df[col].mean()
            row[col + " median"] = df[col].median()
    stats.loc[-1] = row
    stats.index = stats.index + 1
stats = stats[['option', 'Time (ms) mean', 'Propagation x10^3 mean', 'Propagation x10^3 median', 'Sync assign x10^3 mean', 'Sync assign x10^3 median']]
stats.to_markdown("tmp.md")

SAT dataset with 1000 problems with 100 variables.
